# Working with USB Hardware in v4.0

® *Copyright Bimea 2024-2026*

This notebook shows how v4.0 simplifies USB device management while still providing low-level access when needed.

## What Changed in v4.0

- **Automatic detection**: `Megamicros()` finds USB hardware automatically
- **Explicit USB source**: Use `usb=True` to force USB mode
- **Low-level access**: Still available via `megamicros.usb.Usb` for advanced users
- **Better error handling**: Clear exceptions when hardware is not found
- **Multi-device support**: Automatically selects first available device

Let's explore both the high-level (recommended) and low-level approaches.

In [ ]:
import megamicros
from megamicros.log import log

# Set log level to INFO to see device detection details
log.setLevel("INFO")

print(f"Megamicros version: {megamicros.__version__}")

## Method 1: Automatic Detection (Recommended v4.0)

The `Megamicros` class automatically detects and uses USB hardware if available.

In [ ]:
from megamicros import Megamicros

# Create antenna - automatically finds USB hardware
antenna = Megamicros()

info = antenna.infos
print(f"✓ Source type: {info['source_type']}")
print(f"✓ Hardware: {info.get('hardware', 'Unknown')}")
print(f"✓ Available MEMS: {len(antenna.available_mems)}")

# If no USB hardware is found, it falls back to RandomDataSource
# This allows testing code without physical hardware!

## Method 2: Explicit USB Source

Force USB mode explicitly - raises an exception if hardware is not found.

In [ ]:
try:
    # Explicitly request USB source
    antenna_usb = Megamicros(usb=True)
    
    info = antenna_usb.infos
    print("✓ USB hardware successfully connected!")
    print(f"  Product ID: {info.get('product_id', 'N/A')}")
    print(f"  Available MEMS: {len(antenna_usb.available_mems)}")
    print(f"  Max sampling frequency: {info.get('max_sampling_frequency', 'N/A')} Hz")
    
except Exception as e:
    print(f"⚠ USB hardware not available: {e}")
    print("💡 Make sure:")
    print("   1. Megamicros hardware is plugged in")
    print("   2. USB permissions are configured (Linux/macOS)")
    print("   3. Device is not used by another process")

## Method 3: Low-Level USB Access (Advanced)

For advanced users who need direct control over USB transfers, the low-level `Usb` class is still available.

**⚠ Warning**: Only use this ifconfigure custom USB behaviors. Most users should use the high-level API above.

In [ ]:
from megamicros.usb import Usb

# Megamicros USB device identifiers
VENDOR_ID = 0xFE27
PRODUCT_IDS = {
    0xAC00: "Mu32-usb2",
    0xAC01: "Mu32-usb3 / Mu256",
    0xAC02: "Mu1024",
    0xAC03: "Mu32-usb3 (alternate)"
}

# Check for any Megamicros device
device_found = False
detected_product_id = None

for product_id, device_name in PRODUCT_IDS.items():
    if Usb.checkDeviceByVendorProduct(vendor_id=VENDOR_ID, product_id=product_id):
        print(f"✓ Found: {device_name} (Product ID: 0x{product_id:04X})")
        device_found = True
        detected_product_id = product_id
        break

if not device_found:
    print("⚠ No Megamicros USB device found")
    print("Available product IDs:", [f"0x{pid:04X}" for pid in PRODUCT_IDS.keys()])

### Low-Level USB Operations

If you found a device above, you can perform low-level operations:

In [ ]:
if device_found:
    try:
        # Initialize USB connection
        usb_device = Usb()
        usb_device.open(
            vendor_id=VENDOR_ID,
            product_id=detected_product_id,
            bus_address=0x00,
            endpoint_in=0x81
        )
        
        # Claim interface for exclusive access
        usb_device.claim()
        
        print("✓ USB device opened and claimed")
        print(f"  Endpoint IN: 0x81")
        print(f"  Transfer buffer size: {usb_device.buffer_size} bytes")
        
        # Release interface when done
        usb_device.release()
        usb_device.close()
        
        print("✓ USB device released and closed")
        
    except Exception as e:
        print(f"⚠ Low-level USB error: {e}")
else:
    print("⚠ Skipping low-level operations (no device found)")

## USB Configuration in v4.0

You can pass USB-specific configuration via `UsbConfig`:

In [ ]:
from megamicros.core.config import UsbConfig

# Create custom USB configuration
usb_config = UsbConfig(
    vendor_id=0xFE27,
    product_id=0xAC03,  # Mu32-usb3
    queue_size=16,      # Increase buffer (default: 8)
    queue_timeout=2000  # 2 second timeout (default: 1000ms)
)

print("USB Configuration:")
print(f"  Vendor ID: 0x{usb_config.vendor_id:04X}")
print(f"  Product ID: 0x{usb_config.product_id:04X}")
print(f"  Queue size: {usb_config.queue_size} frames")
print(f"  Timeout: {usb_config.queue_timeout}ms")

## Testing Without Hardware

One of the biggest advantages of v4.0 is the ability to test code without physical hardware.

If no USB device is found, `Megamicros()` automatically falls back to `RandomDataSource`:

In [ ]:
import numpy as np

# This works whether you have hardware or not!
antenna = Megamicros()

antenna.run(
    mems=[0, 1, 2, 3],
    duration=0.5,
    sampling_frequency=50000
)

# Collect frames
signal_segments = []
for frame in antenna:
    signal_segments.append(frame)

antenna.wait()

# Concatenate
all_data = np.concatenate(signal_segments, axis=1)
print(f"✓ Collected signal: {all_data.shape}")
print(f"  Source type: {antenna.infos['source_type']}")
print(f"  Real hardware: {'Yes' if antenna.infos['source_type'] == 'usb' else 'No (simulated)'}")

## Troubleshooting USB Connections

### Linux
If you encounter permission errors on Linux, install udev rules:

```bash
sudo tee /etc/udev/rules.d/99-megamicros-devices.rules > /dev/null << 'EOF'
# Megamicros devices
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac00", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac01", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac02", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="fe27", ATTR{idProduct}=="ac03", MODE="0666"
EOF

sudo udevadm control --reload-rules
sudo udevadm trigger
```

### macOS
Ensure you have the correct permissions:

```bash
# Check if device is visible
system_profiler SPUSBDataType | grep -i megamicros

# If not visible, try unplugging and replugging the device
```

### Windows
No special configuration needed - USB should work out of the box.

### Device Already in Use
If another process is using the device:

```python
from megamicros.exception import MuException

try:
    antenna = Megamicros(usb=True)
except MuException as e:
    print(f"Error: {e}")
    print("Try closing other applications using the device")
```

## Key Takeaways

✅ **Recommended**: Use `Megamicros()` for automatic USB detection  
✅ **Explicit mode**: Use `Megamicros(usb=True)` to enforce USB  
✅ **Low-level access**: Use `megamicros.usb.Usb` for advanced control  
✅ **Hardware-free testing**: Auto-falls back to RandomDataSource  
✅ **Platform support**: Linux, macOS, Windows (with udev on Linux)  
✅ **Multi-device**: Automatically selects first available device  

**Next**: See Notebook 03 for advanced Megamicros features!